In [ ]:
# hide
import gzip
import json
from functools import cache

import numpy as np
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
# collapse
SECONDS_PER_BEAT = 0.6                           # a tempo of 100 beats per minute
PITCH_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
CHORD_SHAPES = {"maj": [0, 4, 7], "min": [0, 3, 7], "7": [0, 4, 7, 10],
                "maj7": [0, 4, 7, 11], "min7": [0, 3, 7, 10]}   # semitones above the root


def pluck(pitch: float, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    """A Karplus-Strong plucked string (Chapter 12) that decays by 60 dB in `duration`
    seconds. An all-pass filter (Chapter 13) in the loop tunes it precisely."""
    P = f_s / pq.helper.pitch_to_frequency(pitch)   # the period, in (fractional) samples
    M = int(P - 0.6)                                 # the delay line (the low-pass adds 1/2 sample)
    D = P - 0.5 - M                                  # the all-pass adds the remaining D samples
    c = (1 - D) / (1 + D)                            # the all-pass gain for a delay of D samples
    g = 10 ** (-3 * P / (f_s * duration))            # decay by 60 dB over the duration
    x = np.zeros(int(duration * f_s))
    x[:M] = np.random.uniform(-1.0, 1.0, M)          # excitation: a burst of white noise
    # The whole loop is one recursive filter (Chapter 9), which SciPy runs quickly:
    # y[n] = x[n] + c x[n-1] - c y[n-1] + (g/2) (c y[n-M] + (1 + c) y[n-M-1] + y[n-M-2])
    a = np.zeros(M + 3)
    a[:2] = [1.0, c]
    a[M:] -= 0.5 * g * np.array([c, 1 + c, 1.0])
    return pq.Audio(lfilter([1.0, c], a, x), f_s)


def piano(pitch: float, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    """A piano-like tone by additive synthesis (Chapter 3): six harmonics that decay
    exponentially, the higher ones faster, with a quick fade in and out to avoid clicks."""
    t = np.arange(int(duration * f_s)) / f_s
    f_0 = pq.helper.pitch_to_frequency(pitch)
    x = sum(np.sin(2 * np.pi * k * f_0 * t) * np.exp(-2 * k * t) / k for k in range(1, 7))
    fade = np.clip(np.minimum(t / 0.005, (duration - t) / 0.05), 0, 1)
    return pq.Audio(0.5 * x * fade, f_s)


@cache
def guitar(pitch: float, duration: float) -> pq.Audio:
    """A plucked string for the chords. To save time, each string is plucked just once
    (the first time it is needed), and then reused."""
    return 0.6 * pluck(pitch, duration)


def band(part: str, pitch: float, duration: float, **kwargs) -> pq.Audio:
    """The instrument for a lead sheet: a piano for the melody, a guitar for the chords."""
    return piano(pitch, duration) if part == "melody" else guitar(pitch, duration)


def lead_sheet_to_score(melody: list, chords: list, max_beats: float = 64) -> pq.Score:
    """A score, with times in seconds, of the first max_beats beats of a lead sheet. Each
    melody note lasts until the next one (at most two beats). Each chord is strummed, low
    to high, when it starts, and again on every other beat until the next chord."""
    events = []
    for i, (beat, pitch) in enumerate(melody):
        end = melody[i + 1][0] if i + 1 < len(melody) else beat + 1
        events.append((beat, {"part": "melody", "pitch": pitch,
                              "duration": min(end - beat, 2) * SECONDS_PER_BEAT}))
    for i, (beat, label) in enumerate(chords):
        root, family = label.split(":")
        bass = 48 + PITCH_NAMES.index(root)          # the root, between C3 and B3
        end = chords[i + 1][0] if i + 1 < len(chords) else beat + 4
        strums = [beat] + [b for b in range(int(beat) + 1, int(np.ceil(end))) if b % 2 == 0]
        for strum in strums:
            for j, interval in enumerate(CHORD_SHAPES[family]):
                events.append((strum + 0.03 * j, {"part": "chord", "pitch": bass + interval,
                                                  "duration": 2.0}))
    return pq.Score([(beat * SECONDS_PER_BEAT, kwargs) for beat, kwargs in events
                     if 0 <= beat < max_beats])

In [ ]:
with gzip.open("../assets/TheorytabSimple.json.gz", "rt") as f:
    dataset = json.load(f)                 # {id: {"split", "youtube_url", "melody", "chords"}}
print(f"{len(dataset)} songs")
song = dataset[np.random.choice([i for i in dataset if dataset[i]["split"] == "TRAIN"])]
print("Original:", song["youtube_url"])
print("Melody [beat, pitch]:", song["melody"][:6], "...")
print("Chords [beat, label]:", song["chords"][:4], "...")
pq.play(lead_sheet_to_score(song["melody"], song["chords"]).render(band), normalize=True)

In [ ]:
# collapse
LOWEST, HIGHEST = 48, 84                         # the range of melody pitches, C3 to C6
VOCAB = (["<BOS>", "<EOS>", "BAR"]               # start of song, end of song, new bar
         + [f"CHORD_{root}:{family}" for root in PITCH_NAMES for family in CHORD_SHAPES]
         + [f"NOTE_{pitch}" for pitch in range(LOWEST, HIGHEST + 1)]
         + [f"SHIFT_{k}" for k in range(1, 16)])  # wait k sixteenth notes
INDEX = {token: i for i, token in enumerate(VOCAB)}


def events_to_tokens(melody: list, chords: list) -> list[int]:
    """A lead sheet as token indices: <BOS>, then each bar (of 4 beats) as a BAR token
    followed by its chords and notes in time order (a chord before a note at the same
    time), with SHIFT_k tokens that wait k sixteenth notes, then <EOS>. Onsets are
    rounded to sixteenths, and pitches are moved by octaves into range."""
    events = {}
    for beat, label in chords:
        events[round(4 * beat), 0] = f"CHORD_{label}"
    for beat, pitch in melody:
        while pitch < LOWEST:
            pitch += 12
        while pitch > HIGHEST:
            pitch -= 12
        events[round(4 * beat), 1] = f"NOTE_{pitch}"
    tokens, bar, position = ["<BOS>"], -1, 0
    for (sixteenth, _), token in sorted(events.items()):
        while bar < sixteenth // 16:              # start a new bar
            tokens.append("BAR")
            bar, position = bar + 1, 0
        if position < sixteenth % 16:             # wait until this event
            tokens.append(f"SHIFT_{sixteenth % 16 - position}")
            position = sixteenth % 16
        tokens.append(token)
    tokens.append("<EOS>")
    return [INDEX[token] for token in tokens]


def tokens_to_events(tokens: list[int]) -> tuple[list, list]:
    """The melody ([beat, pitch] pairs) and chords ([beat, label] pairs) of a token
    sequence, up to the first <EOS>."""
    melody, chords, bar, position = [], [], -1, 0
    for token in tokens:
        kind, _, value = VOCAB[token].partition("_")
        if kind == "<EOS>":
            break
        elif kind == "BAR":
            bar, position = bar + 1, 0
        elif kind == "SHIFT":
            position += int(value)
        elif kind == "CHORD":
            chords.append([4 * bar + position / 4, value])
        elif kind == "NOTE":
            melody.append([4 * bar + position / 4, int(value)])
    return melody, chords


assert list(np.load("../assets/lead-sheet-rnn.npz")["vocab"]) == VOCAB  # the model's tokens

In [ ]:
weights = np.load("../assets/lead-sheet-rnn.npz")       # trained offline on the TRAIN songs
params = {name: weights[name] for name in ["E", "W_x", "W_h", "b_h", "W_y", "b_y"]}


def rnn_step(params: dict[str, np.ndarray], token: int,
             h: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Read one token and update the hidden state h. Returns the new hidden state, and
    the probability of each token in the vocabulary being the next one."""
    h = np.tanh(params["E"][token] @ params["W_x"] + h @ params["W_h"] + params["b_h"])
    z = h @ params["W_y"] + params["b_y"]
    p = np.exp(z - z.max())                               # a softmax
    return h, p / p.sum()


print(f"{len(VOCAB)} tokens. The song above begins:")
print([VOCAB[token] for token in events_to_tokens(song["melody"], song["chords"])[:12]])
print(f"Average negative log-likelihood of the VALID songs: "
      f"{weights['valid_bits_per_token']:.2f} bits per token")

In [ ]:
temperature = 0.9                          # lower is more predictable, higher more surprising
tokens = [INDEX["<BOS>"], INDEX["BAR"]]    # start at the beginning of a bar
h, _ = rnn_step(params, tokens[0], np.zeros(len(params["b_h"])))
while tokens.count(INDEX["BAR"]) <= 16:    # generate 16 bars
    h, p = rnn_step(params, tokens[-1], h)
    p[INDEX["<EOS>"]] = 0                  # don't let the song end early
    p = p ** (1 / temperature)
    tokens.append(np.random.choice(len(VOCAB), p=p / p.sum()))
melody, chords = tokens_to_events(tokens)
print("Generated:", [VOCAB[token] for token in tokens[:16]], "...")
pq.play(lead_sheet_to_score(melody, chords).render(band), normalize=True)